# Train a story-writing GPT in 15 minutes

**A live demo for CPSC 1710, Lab 5. It needs a GPU runtime; it was written for an A100 (Colab Pro).**

The model in `gpt_dev.ipynb` reads one character at a time and learns from one megabyte of Shakespeare. Here we keep the same kind of model and the same training loop, and change two things:

- **the text:** TinyStories, a collection of very short stories written with a small child's vocabulary;
- **the tokens:** word pieces, the kind of token the models you call through an API use.

Then we train for a fixed number of minutes. Every 90 seconds the model is given the same opening words and asked to continue, so you can watch what it has learned so far.

**How to run it:** run the cells from top to bottom. The first three prepare the data and take a few minutes. The cell marked **Train** is the timed run.

*Drafted by Claude (Anthropic), based on the direction of Xiuye Chen. Model code from Andrej Karpathy's nanoGPT.*

In [ ]:
# Settings. Change these and run the notebook again.
MINUTES      = 15                                   # how long the training cell runs
SAMPLE_EVERY = 90                                   # seconds between samples
PROMPT       = "Once upon a time, a little robot"   # the same opening every time, so progress is visible
DATA_TOKENS  = 300_000_000                          # how much of the text to prepare, in tokens

# The dials, as on the tour's "The dials a person set" slide.
N_LAYER, N_HEAD, N_EMBD = 6, 6, 384                 # blocks, attention patterns per block, numbers per token
BLOCK_SIZE, BATCH_SIZE  = 256, 128                  # how far back it looks (in tokens), stories per step
TRAIN_FILE, VALID_FILE  = "TinyStoriesV2-GPT4-train.txt", "TinyStoriesV2-GPT4-valid.txt"

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "none. Choose a GPU runtime (Runtime > Change runtime type) before continuing.")

## 1. Get the stories

The training file is about 2 GB of plain text. Here are two of the stories in it.

In [ ]:
%pip -q install tiktoken huggingface_hub
!git clone -q https://github.com/karpathy/nanoGPT.git 2>/dev/null || true

from huggingface_hub import hf_hub_download
train_txt = hf_hub_download("roneneldan/TinyStories", TRAIN_FILE, repo_type="dataset")
valid_txt = hf_hub_download("roneneldan/TinyStories", VALID_FILE, repo_type="dataset")

with open(valid_txt, encoding="utf-8") as f:
    print(f.read(1500).split("<|endoftext|>")[0].strip(), "\n\n...and thousands more like it.")

## 2. Turn the text into tokens

Same step as Move 1 of the tour, with a different vocabulary: about 50,000 word pieces in place of 65 characters. This is the tokenizer GPT-2 used.

In [ ]:
import os, time, numpy as np, tiktoken
enc = tiktoken.get_encoding("gpt2")

for word in ["robot", "Once upon a time", "unbelievable"]:
    print(f"{word!r:22} -> {enc.encode(word)}  pieces: {[enc.decode([t]) for t in enc.encode(word)]}")

def tokenize(text_path, out_path, budget):
    """Read stories, turn each into token ids, and save them all in one long file of numbers."""
    if os.path.exists(out_path):
        return np.memmap(out_path, dtype=np.uint16, mode="r")
    start, total, batch, story = time.time(), 0, [], []
    def flush(out):
        ids = enc.encode_ordinary_batch(batch, num_threads=os.cpu_count() or 2)
        arr = np.concatenate([np.array(x + [enc.eot_token], dtype=np.uint16) for x in ids])
        arr.tofile(out)
        return len(arr)
    with open(text_path, encoding="utf-8") as f, open(out_path, "wb") as out:
        for line in f:
            if line.strip() == "<|endoftext|>":
                if story: batch.append("".join(story).strip())
                story = []
                if len(batch) >= 20000:
                    total += flush(out); batch = []
                    if total >= budget: break
            else:
                story.append(line)
        if story and total < budget: batch.append("".join(story).strip())
        if batch: total += flush(out)
    print(f"{os.path.basename(out_path)}: {total:,} tokens in {time.time() - start:.0f} s")
    return np.memmap(out_path, dtype=np.uint16, mode="r")

train_data = tokenize(train_txt, "train.bin", DATA_TOKENS)
val_data   = tokenize(valid_txt, "val.bin", 5_000_000)
print(f"training tokens: {len(train_data):,}   held-back tokens: {len(val_data):,}")
print("The Shakespeare file was about 1,000,000 characters.")

## 3. Build the model

The same design as the notebook's model: token and position embeddings, a stack of blocks (attention, then a small network), and one score for every token in the vocabulary. Only the dials are larger.

In [ ]:
import sys, math
sys.path.insert(0, "nanoGPT")
from model import GPT, GPTConfig

torch.manual_seed(1710)
config = GPTConfig(block_size=BLOCK_SIZE, vocab_size=enc.n_vocab, n_layer=N_LAYER, n_head=N_HEAD, n_embd=N_EMBD, dropout=0.0, bias=False)
model = GPT(config).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}   (the notebook's model had 209,729)")

## 4. Train

This is the timed run. The loop is the one from the tour: grab some text, guess every next token, measure the loss, nudge every parameter, repeat.

Before any training the loss should be near 10.8. That is what guessing blindly among 50,000 tokens scores. Watch the loss fall and the samples change.

In [ ]:
# ---- Train ----
use_amp  = device == "cuda"
amp_type = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
ctx      = torch.autocast(device_type="cuda", dtype=amp_type) if use_amp else torch.autocast(device_type="cpu", enabled=False)
scaler   = torch.amp.GradScaler("cuda", enabled=(use_amp and amp_type == torch.float16))
if use_amp:
    torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True

def get_batch(data):
    ix = torch.randint(len(data) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([torch.from_numpy(data[i:i + BLOCK_SIZE].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + 1 + BLOCK_SIZE].astype(np.int64)) for i in ix])
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

@torch.no_grad()
def held_back_loss(batches=20):
    model.eval()
    losses = []
    for _ in range(batches):
        x, y = get_batch(val_data)
        with ctx:
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

@torch.no_grad()
def write(prompt, tokens=90):
    model.eval()
    idx = torch.tensor([enc.encode(prompt)], device=device)
    with ctx:
        out = model.generate(idx, max_new_tokens=tokens, temperature=0.8, top_k=50)
    model.train()
    return enc.decode(out[0].tolist()).split("<|endoftext|>")[0].strip()

def report(elapsed, tokens_seen):
    loss = held_back_loss()
    history.append((elapsed / 60, tokens_seen, loss))
    print(f"\n[{int(elapsed // 60):2d}:{int(elapsed % 60):02d}]  tokens read so far: {tokens_seen:>13,}   loss on held-back stories: {loss:.2f}")
    print(write(PROMPT))

LEARNING_RATE = 5e-4                                 # the size of each nudge
optimizer = model.configure_optimizers(weight_decay=0.1, learning_rate=LEARNING_RATE, betas=(0.9, 0.95), device_type=device)
train_model = model
if device == "cuda":
    try:
        train_model = torch.compile(model)          # faster steps after a one-off wait of about a minute
    except Exception as e:
        print("Running without torch.compile:", e)

history, budget, step, tokens_seen = [], MINUTES * 60, 0, 0
print("Before any training:"); report(0, 0)

x, y = get_batch(train_data)
with ctx:                                            # one untimed practice step, so the clock starts after the one-off compile
    _, loss = train_model(x, y)
scaler.scale(loss).backward(); optimizer.zero_grad(set_to_none=True)
print("\nTraining for", MINUTES, "minutes...")
start = last = time.time()

while True:
    elapsed = time.time() - start
    if elapsed >= budget:
        break
    frac = elapsed / budget                          # learning rate: quick warm-up, then a smooth decline
    lr = LEARNING_RATE * min(1.0, (step + 1) / 200) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * frac)))
    for group in optimizer.param_groups:
        group["lr"] = lr

    x, y = get_batch(train_data)                     # grab some text
    with ctx:
        _, loss = train_model(x, y)                  # guess every next token, and measure the loss
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()                    # which way should each parameter move?
    scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer); scaler.update()          # nudge them all
    step += 1; tokens_seen += x.numel()

    if time.time() - last >= SAMPLE_EVERY:
        report(time.time() - start, tokens_seen); last = time.time()

total = time.time() - start
report(total, tokens_seen)
print(f"\nDone: {step:,} steps, {tokens_seen:,} tokens read, {tokens_seen / total:,.0f} tokens per second.")
print(f"That is {tokens_seen / len(train_data):.2f} passes over the prepared text.")
torch.save({"model": model.state_dict(), "config": config.__dict__}, "tinystories-gpt.pt")

## 5. The loss, over the run

In [ ]:
import matplotlib.pyplot as plt
mins, _, losses = zip(*history)
plt.figure(figsize=(7, 3.6))
plt.plot(mins, losses, marker="o")
plt.axhline(math.log(enc.n_vocab), ls="--", lw=1, color="gray"); plt.text(mins[-1], math.log(enc.n_vocab), " blind guessing", va="bottom", ha="right", color="gray")
plt.xlabel("minutes of training"); plt.ylabel("loss on held-back stories (lower is better)"); plt.grid(alpha=.3); plt.show()

## 6. Give it your own opening

Stay inside the world it was trained on: simple words, short sentences, children's stories. Then try something from outside that world and see what happens.

In [ ]:
for opening in ["One day, a cat and a dog", "Lily was sad because", "The capital of France is"]:
    print(">>", write(opening, tokens=110), "\n")

## What to take from this

- **The recipe did not change.** This is the loop from `gpt_dev.ipynb`. More text, word-piece tokens and more parameters are what changed.
- **The model knows only its training text.** It can write a story about a cat. It cannot tell you the capital of France, because nothing like that was in the stories.
- **Scale is the remaining difference.** Compare the number of tokens this model read, printed above, with the roughly 4 trillion that the open model OLMo 2 read during its training. The models behind chatbots are trained in the same way, for far longer, on far more text, and then trained further on conversations.

To go on from here, the optional notebook *From the tiny GPT to the models you call* loads a TinyStories model that was trained for much longer, and OLMo 2 at four points in its training.